In [41]:
import os
import json
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
import faiss
from groq import Groq

print("All imports successful!")

BASE_DIR    = Path(r"C:\Users\ishas\OneDrive\Desktop\ML\Ecolens")
CHATBOT_DIR = BASE_DIR / "src" / "chatbot_data"
CHATBOT_DIR.mkdir(parents=True, exist_ok=True)

print("Chatbot data folder:", CHATBOT_DIR)

All imports successful!
Chatbot data folder: C:\Users\ishas\OneDrive\Desktop\ML\Ecolens\src\chatbot_data


In [48]:
# This is your RAG knowledge base
# Each entry has a category, class name, and detailed disposal information
# The chatbot will search through this when a user asks a question

WASTE_KNOWLEDGE_BASE = [

    # ── GLASS ──────────────────────────────────────────────────
    {
        "class"   : "glass",
        "category": "disposal",
        "text"    : """Glass disposal and recycling guide:
        Glass bottles, jars, and containers are 100% recyclable and can be 
        recycled endlessly without losing quality. Rinse glass containers 
        before recycling. Remove metal lids and recycle separately. 
        Do not include broken glass in recycling bins — wrap in newspaper 
        and place in general waste for safety. Window glass, mirrors, 
        and Pyrex cannot be recycled with regular glass as they have 
        different melting points. In India, glass can be given to 
        kabadiwala (scrap dealers) who pay for it by weight."""
    },
    {
        "class"   : "glass",
        "category": "environment",
        "text"    : """Environmental impact of glass waste:
        Glass takes over 1 million years to decompose in landfill. 
        Recycling one glass bottle saves enough energy to power a 
        computer for 25 minutes. Glass production requires silica sand 
        mining which damages ecosystems. Using recycled glass reduces 
        air pollution by 20% and water pollution by 50% compared to 
        virgin glass production."""
    },

    # ── HAZARDOUS ───────────────────────────────────────────────
    {
        "class"   : "hazardous",
        "category": "disposal",
        "text"    : """Hazardous waste disposal — batteries and chemicals:
        Never throw batteries in regular trash or recycling bins. 
        Batteries contain lead, mercury, cadmium and lithium which 
        contaminate soil and groundwater. In India, take batteries to 
        authorized collection centers. Many electronics stores like 
        Croma and Reliance Digital accept old batteries. 
        For household chemicals like paint, pesticides, and cleaning 
        products — never pour down drains. Contact your local 
        municipal corporation for hazardous waste collection drives. 
        CPCB (Central Pollution Control Board) regulates hazardous 
        waste disposal in India."""
    },
    {
        "class"   : "hazardous",
        "category": "environment",
        "text"    : """Why hazardous waste is dangerous:
        Batteries leaching into soil can contaminate groundwater for 
        decades. One AA battery can pollute 1000 liters of water. 
        Mercury from batteries accumulates in fish and enters the 
        human food chain causing neurological damage. CFL bulbs contain 
        mercury vapor — break one in a room and ventilate immediately. 
        Never burn hazardous waste as it releases toxic fumes."""
    },

    # ── METAL ───────────────────────────────────────────────────
    {
        "class"   : "metal",
        "category": "disposal",
        "text"    : """Metal recycling guide — cans, foil, scrap:
        Aluminium cans are the most valuable recyclable material. 
        Recycling aluminium uses 95% less energy than producing new aluminium. 
        Steel cans, tins and containers are fully recyclable. 
        Clean metal containers before recycling — food residue contaminates 
        recycling batches. Aluminium foil can be recycled if clean — 
        scrunch multiple pieces into a ball before recycling. 
        In India, scrap metal dealers (kabadiwala) actively buy metal 
        by weight. Old appliances can be given to authorized e-waste 
        recyclers or scrap dealers."""
    },
    {
        "class"   : "metal",
        "category": "environment",
        "text"    : """Environmental impact of metal waste:
        Mining metal ores destroys habitats and causes severe pollution. 
        Recycling one aluminium can saves enough energy to run a TV 
        for 3 hours. Steel recycling saves 1.1 tonnes of CO2 per tonne 
        of steel recycled. Metals do not biodegrade — they persist in 
        environment for centuries. Rusting metal leaches iron oxide 
        into soil affecting plant growth."""
    },

    # ── PAPER AND CARDBOARD ─────────────────────────────────────
    {
        "class"   : "paper_cardboard",
        "category": "disposal",
        "text"    : """Paper and cardboard recycling guide:
        Flatten cardboard boxes before recycling to save space. 
        Remove tape, staples and plastic windows from envelopes. 
        Pizza boxes contaminated with grease cannot be recycled — 
        tear off the clean top and recycle that, compost the greasy bottom. 
        Shredded paper should be placed in a paper bag before recycling 
        as it jams sorting machines. Newspapers, magazines, office paper, 
        cardboard packaging are all recyclable. Thermal paper receipts 
        contain BPA and cannot be recycled — place in general waste. 
        In India, newspapers and cardboard are highly valued by 
        kabadiwala — approximately Rs 10-15 per kg."""
    },
    {
        "class"   : "paper_cardboard",
        "category": "reuse",
        "text"    : """Creative reuse of paper and cardboard:
        Cardboard boxes can be reused for storage, moving, or craft projects. 
        Newspapers make excellent packing material, shelf liners, and 
        can be used for cleaning glass windows. Old magazines can be 
        donated to hospitals, schools, or community centers. 
        Cardboard tubes from toilet rolls make excellent seed starters 
        for gardening. Paper bags can replace plastic bags for shopping."""
    },

    # ── ORGANIC BIODEGRADABLE ───────────────────────────────────
    {
        "class"   : "organic_biodegradable",
        "category": "disposal",
        "text"    : """Organic and biodegradable waste disposal:
        Food waste and organic material should ideally be composted 
        rather than sent to landfill. Home composting: collect fruit 
        peels, vegetable scraps, coffee grounds, eggshells, and garden 
        waste in a compost bin. Avoid composting meat, dairy, and oily 
        food as they attract pests. Compost takes 2-6 months to mature 
        into rich fertilizer. In India, many municipalities have 
        wet waste (green bin) and dry waste (blue bin) segregation. 
        Wet/organic waste goes in the green bin for municipal composting. 
        Biogas plants can convert organic waste to cooking gas — 
        several Indian cities have community biogas plants."""
    },
    {
        "class"   : "organic_biodegradable",
        "category": "environment",
        "text"    : """Why composting matters:
        When organic waste goes to landfill it produces methane — 
        a greenhouse gas 25x more potent than CO2. 
        Composting returns nutrients to soil, reducing need for 
        chemical fertilizers. India generates 62 million tonnes of 
        solid waste annually — 50% is organic. Composting could 
        eliminate half of India's landfill waste. 
        Vermicomposting using earthworms produces even richer compost 
        and can be done in a small apartment."""
    },

    # ── NON RECYCLABLE TRASH ────────────────────────────────────
    {
        "class"   : "non_recyclable_trash",
        "category": "disposal",
        "text"    : """Non-recyclable waste disposal:
        Items like chip packets, multilayer plastic packaging, 
        broken toys, used diapers, sanitary waste, and contaminated 
        materials cannot be recycled and must go to general waste. 
        Minimize non-recyclable waste by choosing products with 
        recyclable packaging. Some non-recyclables like clothes and 
        shoes can be donated instead of discarded. 
        Styrofoam/thermocol is technically recyclable but rarely 
        accepted — check local facilities. Co-processing in cement 
        kilns is used in India to dispose of non-recyclable waste 
        at high temperatures safely."""
    },
    {
        "class"   : "non_recyclable_trash",
        "category": "reduce",
        "text"    : """Reducing non-recyclable waste:
        The best approach to non-recyclable waste is to not generate it. 
        Choose loose produce over packaged. Carry reusable bags, 
        bottles and containers. Choose products with minimal packaging. 
        Buy in bulk to reduce per-unit packaging. 
        Repair items instead of discarding — clothes, electronics, 
        furniture. Choose quality items that last longer over cheap 
        disposable alternatives."""
    },

    # ── RECYCLABLE PLASTIC ──────────────────────────────────────
    {
        "class"   : "recyclable_plastic",
        "category": "disposal",
        "text"    : """Recyclable plastic guide:
        Plastics are numbered 1-7. Types 1 (PET — water bottles) 
        and 2 (HDPE — milk jugs, shampoo bottles) are most widely recycled. 
        Type 5 (PP — yogurt containers, straws) is increasingly accepted. 
        Types 3, 6, 7 are rarely recyclable — check local guidelines. 
        Rinse plastic containers — food contamination ruins entire 
        recycling batches. Remove labels if possible. 
        Do not recycle plastic bags in regular bins — they jam machinery. 
        Many supermarkets have plastic bag drop-off points. 
        In India, plastic waste management rules 2016 mandate that 
        producers take responsibility for plastic waste collection."""
    },
    {
        "class"   : "recyclable_plastic",
        "category": "environment",
        "text"    : """Plastic pollution facts:
        Only 9% of all plastic ever produced has been recycled globally. 
        Plastic takes 400-1000 years to decompose. 
        8 million tonnes of plastic enters oceans every year. 
        Microplastics have been found in human blood, breast milk, 
        and placentas. India banned single-use plastics in 2022 
        including straws, plates, cups, and carry bags under 75 microns. 
        Recycling one tonne of plastic saves 5774 kWh of energy, 
        16.3 barrels of oil, and 30 cubic yards of landfill space."""
    },

    # ── GENERAL INDIA SPECIFIC ──────────────────────────────────
    {
        "class"   : "general",
        "category": "india",
        "text"    : """Waste management in India — key facts:
        Swachh Bharat Mission launched in 2014 aims for clean India. 
        Source segregation is mandatory — wet waste (green bin), 
        dry waste (blue bin), hazardous waste (red bin). 
        Extended Producer Responsibility (EPR) makes manufacturers 
        responsible for collecting their packaging waste. 
        CPCB regulates hazardous and biomedical waste. 
        Kabadiwala (informal waste collectors) play a crucial role — 
        India's informal recycling sector handles 90% of recyclables. 
        ITC's WOW (Well-being Out of Waste) program collects 
        and recycles over 120,000 tonnes of waste annually."""
    },
    {
        "class"   : "general",
        "category": "tips",
        "text"    : """General waste reduction tips:
        Follow the 5R hierarchy: Refuse, Reduce, Reuse, Recycle, Rot.
        Refuse unnecessary items — say no to single use plastics.
        Reduce consumption overall — buy only what you need.
        Reuse items as many times as possible before discarding.
        Recycle what cannot be reused — clean and sort properly.
        Rot — compost organic waste instead of landfilling.
        A zero-waste lifestyle is achievable with small consistent changes.
        Track your waste for one week to identify biggest waste categories."""
    }
    
]

print(f"Knowledge base created: {len(WASTE_KNOWLEDGE_BASE)} entries")
for entry in WASTE_KNOWLEDGE_BASE:
    print(f"  {entry['class']:<25} — {entry['category']}")

Knowledge base created: 16 entries
  glass                     — disposal
  glass                     — environment
  hazardous                 — disposal
  hazardous                 — environment
  metal                     — disposal
  metal                     — environment
  paper_cardboard           — disposal
  paper_cardboard           — reuse
  organic_biodegradable     — disposal
  organic_biodegradable     — environment
  non_recyclable_trash      — disposal
  non_recyclable_trash      — reduce
  recyclable_plastic        — disposal
  recyclable_plastic        — environment
  general                   — india
  general                   — tips


In [43]:
# Load the sentence transformer model
# This model converts text into 384-dimensional vectors
# Similar texts will have similar vectors (close in vector space)

print("Loading sentence transformer model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
# all-MiniLM-L6-v2 is fast, lightweight, and very accurate
# It converts any text into a 384-dimensional vector
print("Model loaded!")

# Extract just the text from each knowledge base entry
texts = [entry["text"] for entry in WASTE_KNOWLEDGE_BASE]

print(f"\nConverting {len(texts)} knowledge chunks to vectors...")
# encode() converts each text string into a 384-dim vector
# This captures the semantic meaning of each chunk
embeddings = embedder.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"Embeddings shape: {embeddings.shape}")
# Should be (16, 384) — 16 chunks, each 384 numbers

# Normalize vectors for cosine similarity search
# After normalization, dot product = cosine similarity
faiss.normalize_L2(embeddings)

# Build FAISS index
# IndexFlatIP = Inner Product (dot product) search
# After L2 normalization this equals cosine similarity
dimension = embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(dimension)
index.add(embeddings.astype(np.float32))

print(f"\nFAISS index built!")
print(f"Total vectors in index: {index.ntotal}")
print(f"Vector dimension      : {dimension}")

# Save index and knowledge base to disk
faiss.write_index(index, str(CHATBOT_DIR / "waste_index.faiss"))

with open(CHATBOT_DIR / "knowledge_base.json", "w") as f:
    json.dump(WASTE_KNOWLEDGE_BASE, f, indent=2)

print(f"\nSaved to {CHATBOT_DIR}")
print("  waste_index.faiss    — FAISS vector index")
print("  knowledge_base.json  — original text chunks")

Loading sentence transformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded!

Converting 16 knowledge chunks to vectors...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (16, 384)

FAISS index built!
Total vectors in index: 16
Vector dimension      : 384

Saved to C:\Users\ishas\OneDrive\Desktop\ML\Ecolens\src\chatbot_data
  waste_index.faiss    — FAISS vector index
  knowledge_base.json  — original text chunks


In [28]:
def retrieve_relevant_chunks(query, embedder, index, knowledge_base, top_k=3):
    """
    Given a user question, find the most relevant
    chunks from the knowledge base.
    
    Steps:
    1. Convert question to vector using same embedder
    2. Search FAISS for most similar vectors
    3. Return top_k most relevant text chunks
    """
    # Convert question to vector
    query_vector = embedder.encode(
        [query],
        convert_to_numpy=True
    )
    
    # Normalize (same as we did for the knowledge base)
    faiss.normalize_L2(query_vector)
    
    # Search FAISS — returns distances and indices
    # distances: how similar (higher = more similar)
    # indices: which chunks matched
    distances, indices = index.search(
        query_vector.astype(np.float32), top_k
    )
    
    # Collect the matched chunks
    results = []
    for i, (idx, dist) in enumerate(zip(indices[0], distances[0])):
        chunk = knowledge_base[idx]
        results.append({
            "rank"      : i + 1,
            "similarity": float(dist),
            "class"     : chunk["class"],
            "category"  : chunk["category"],
            "text"      : chunk["text"]
        })
    
    return results


# Test the retrieval
test_query = "How do I dispose of batteries safely?"
results    = retrieve_relevant_chunks(
    test_query, embedder, index, WASTE_KNOWLEDGE_BASE
)

print(f"Query: '{test_query}'")
print(f"\nTop {len(results)} most relevant chunks:")
print("-" * 50)
for r in results:
    print(f"Rank {r['rank']}: [{r['class']}] {r['category']} "
          f"(similarity: {r['similarity']:.3f})")
    print(f"  Preview: {r['text'][:100]}...")
    print()

Query: 'How do I dispose of batteries safely?'

Top 3 most relevant chunks:
--------------------------------------------------
Rank 1: [hazardous] disposal (similarity: 0.669)
  Preview: Hazardous waste disposal — batteries and chemicals:
        Never throw batteries in regular trash o...

Rank 2: [hazardous] environment (similarity: 0.540)
  Preview: Why hazardous waste is dangerous:
        Batteries leaching into soil can contaminate groundwater f...

Rank 3: [non_recyclable_trash] reduce (similarity: 0.427)
  Preview: Reducing non-recyclable waste:
        The best approach to non-recyclable waste is to not generate ...



In [45]:
from groq import Groq

GROQ_API_KEY = "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"
groq_client  = Groq(api_key=GROQ_API_KEY)
MODEL_NAME   = "llama-3.1-8b-instant"

def generate_answer(query, retrieved_chunks, detected_class=None):
    context = "\n\n".join([
        f"[Source {r['rank']} — {r['class']} | {r['category']}]\n"
        f"{r['text']}"
        for r in retrieved_chunks
    ])

    system_prompt = """You are EcoLens AI, an expert waste management
assistant focused on India. Help people dispose, recycle and reduce waste.
- Give practical, actionable advice
- Mention India-specific options like kabadiwala, Swachh Bharat,
  municipal rules, CPCB guidelines when relevant
- Be concise, friendly and encouraging
- Answer ONLY based on the provided context"""

    user_message = f"""Detected waste class: {detected_class if detected_class else 'Not specified'}

Context:
{context}

Question: {query}

Answer:"""

    response = groq_client.chat.completions.create(
        model    = MODEL_NAME,
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ],
        max_tokens  = 512,
        temperature = 0.3
    )
    return response.choices[0].message.content


# Test
print("Testing Groq...")
test = groq_client.chat.completions.create(
    model    = MODEL_NAME,
    messages = [{"role": "user", "content": "Say hello in one sentence"}],
    max_tokens = 30
)
print("Connected:", test.choices[0].message.content)
print("Model:", MODEL_NAME)
print("Groq + Llama3 ready!")

Testing Groq...
Connected: Hello, how are you today?
Model: llama-3.1-8b-instant
Groq + Llama3 ready!


In [46]:
def ecolens_chatbot(query, detected_class=None):
    print(f"Question       : {query}")
    if detected_class:
        print(f"Detected class : {detected_class}")
    print("-" * 55)
    
    chunks = retrieve_relevant_chunks(
        query, embedder, index,
        WASTE_KNOWLEDGE_BASE, top_k=3
    )
    
    print("Retrieved from knowledge base:")
    for c in chunks:
        print(f"  Rank {c['rank']}: [{c['class']}] "
              f"{c['category']} "
              f"(similarity: {c['similarity']:.3f})")
    print()
    
    answer = generate_answer(query, chunks, detected_class)
    
    print("Answer:")
    print(answer)
    print("\n" + "=" * 55 + "\n")
    
    return answer


# Test 1
ecolens_chatbot(
    query          = "How should I dispose of this item?",
    detected_class = "hazardous"
)

# Test 2
ecolens_chatbot(
    query          = "What happens if I throw batteries in regular trash?",
    detected_class = "hazardous"
)

# Test 3
ecolens_chatbot(
    query = "Can I sell my old newspapers to kabadiwala?"
)

# Test 4
ecolens_chatbot(
    query          = "Can I compost this at home?",
    detected_class = "organic_biodegradable"
)

Question       : How should I dispose of this item?
Detected class : hazardous
-------------------------------------------------------
Retrieved from knowledge base:
  Rank 1: [general] tips (similarity: 0.475)
  Rank 2: [non_recyclable_trash] reduce (similarity: 0.454)
  Rank 3: [hazardous] disposal (similarity: 0.432)

Answer:
Since the detected waste class is hazardous, I'll provide guidance based on the hazardous waste disposal tips.

To dispose of this hazardous item, please follow these steps:

1. **Do not throw it in regular trash or recycling bins** as it can contaminate soil and groundwater.
2. **Check with local authorities**: Contact your local municipal corporation to see if they have any hazardous waste collection drives or designated collection centers in your area.
3. **Visit authorized collection centers**: In India, many electronics stores like Croma and Reliance Digital accept old batteries. You can also check with local hardware stores or pharmacies that sell hazardo

"You can compost organic and biodegradable waste at home. To do this, follow these steps:\n\n1. **Collect the waste**: Gather food waste, fruit peels, vegetable scraps, coffee grounds, eggshells, and garden waste in a compost bin.\n2. **Avoid certain items**: Don't compost meat, dairy, or oily food as they attract pests.\n3. **Create a compost bin**: You can buy a compost bin or make one at home using a wooden or plastic container.\n4. **Add brown and green materials**: Mix equal parts of brown materials (dried leaves, twigs) and green materials (food waste, grass clippings) to create a balanced compost.\n5. **Maintain the bin**: Keep the compost bin moist, turn the compost regularly, and let it mature for 2-6 months.\n\nIn India, you can also use the green bin provided by your municipality for wet waste (organic waste) segregation, which will be composted by the municipal authorities.\n\nRemember, home composting is a great way to reduce landfill waste, produce nutrient-rich fertilize

In [49]:
config = {
    "embedder_model"   : "all-MiniLM-L6-v2",
    "llm_provider"     : "groq",
    "llm_model"        : "llama-3.1-8b-instant",
    "top_k"            : 3,
    "max_output_tokens": 512,
    "temperature"      : 0.3,
    "index_path"       : str(CHATBOT_DIR / "waste_index.faiss"),
    "kb_path"          : str(CHATBOT_DIR / "knowledge_base.json"),
}

with open(CHATBOT_DIR / "chatbot_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Config saved!")
print("\nFiles in chatbot_data/:")
for f in sorted(CHATBOT_DIR.iterdir()):
    size = f.stat().st_size / 1024
    print(f"  {f.name:<30} {size:.1f} KB")

print("\nPhase 5 COMPLETE!")


Config saved!

Files in chatbot_data/:
  chatbot_config.json            0.4 KB
  knowledge_base.json            10.7 KB
  waste_index.faiss              24.0 KB

Phase 5 COMPLETE!
